# Comparación entre Servicios de Salud Metropolitanos

**Propósito:** aplicar la misma preparación y los mismos KPI del `EDA.ipynb` (Servicio de Salud Metropolitano Central) a los demás Servicios de Salud Metropolitanos, para comparar resultados con definiciones idénticas.

**Fuente:** Portal de Transparencia, nómina de personal a contrata, archivos mensuales por servicio.

**Cómo agregar un servicio (por ejemplo, Sur):** copiar sus CSV en `Data/` con el prefijo del servicio (`Sur_TransparenciaActiva_AAAA-MM.csv`) y volver a ejecutar el notebook completo. Los servicios sin archivos se omiten automáticamente.

**Integrantes:** Alessandro Lavezzi - José Saavedra

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sbn
import matplotlib.pyplot as plt
import re 
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## 1. Configuración, lectura e integración

- `SERVICIOS`: patrón de archivos de cada servicio. Centro no tiene prefijo, igual que en el `EDA.ipynb`.
- `MESES_ATIPICOS`: meses que se marcan como atípicos en cada servicio. Centro parte con enero de 2025 (sección 4.3 del `EDA.ipynb`); los demás se completan después de revisar los controles de la sección 2.
- Se usan las mismas funciones de conversión del `EDA.ipynb` (montos, horas extra y fechas), aplicadas por igual a todos los servicios.

In [ ]:
#CONFIGURACIÓN
DATA_DIR = Path("..") / "Data"

SERVICIOS = {
    "Centro": "TransparenciaActiva_*.csv",
    "Norte": "Norte_TransparenciaActiva_*.csv",
    "Oriente": "Oriente_TransparenciaActiva_*.csv",
    "Occidente": "Occidente_TransparenciaActiva_*.csv",
    "Sur": "Sur_TransparenciaActiva_*.csv",
    "SurOriente": "SurOriente_TransparenciaActiva_*.csv",
}
MESES_ATIPICOS = {
    "Centro": ["2025-01"],
    "Sur": ["2025-10"], 
}          

VENTANA_PROPIA = ["Sur"]                          # servicios con meses faltantes: no recortan la ventana común de los demás

UMBRAL_PERSISTENCIA = 75                          # % de meses con horas extra
MESES_MINIMOS = 12                                # meses en la nómina para entrar al KPI de persistencia

NOMBRE_CORTO = {"Médicos cirujanos, farmacéuticos, químico-farmacéuticos, bioquímicos, cirujano-dentistas": "Médicos y afines"}
ORDEN_EST = ["Auxiliar", "Administrativo", "Técnico", "Profesional", "Médicos y afines"]
COLOR = dict(zip(SERVICIOS, plt.cm.tab10.colors))  # color fijo por servicio en todas las figuras

In [ ]:
#FUNCIONES DE PREPARACIÓN (MISMA LÓGICA QUE EDA.ipynb, SECCIÓN 2)
COLS_MONTO = {
    "Remuneración bruta del mes (incluye bonos e incentivos, asig. especiales, horas extras)": "rem_bruta",
    "Remuneración líquida del mes": "rem_liquida",
    "Rem. adicionales del mes (no inc. en rem. bruta)": "rem_adicionales",
    "Remuneración Bonos incentivos del mes (inc. en rem. bruta)": "bonos",
}
COLS_HHEE = {
    "Montos y horas extraordinarias diurnas del mes(inc. en rem. bruta)": "diurnas",
    "Montos y horas extraordinarias nocturnas del mes(inc. en rem. bruta)": "nocturnas",
    "Montos y horas extraordinarias festivas del mes (inc. en rem. bruta)": "festivas",
}
COLS_FECHA = {
    "Fecha de inicio dd/mm/aa": "fecha_inicio",
    "Fecha de término dd/mm/aa": "fecha_termino",
}

def a_monto(serie):
    limpio = (
        serie.str.replace("$", "", regex=False)
             .str.replace(".", "", regex=False)
             .str.strip()
    )
    return pd.to_numeric(limpio, errors="coerce")

def leer_servicio(servicio, patron):
    tablas, archivos = [], []
    for f in sorted(DATA_DIR.glob(patron)):
        t = pd.read_csv(f, sep=";", encoding="latin-1", dtype=str)
        sin_nombre = [c for c in t.columns if c.startswith("Unnamed")]
# el periodo se toma del patrón AAAA-MM: en Sur el nombre termina con el hospital (ej. _HospitalElPeral)
        periodo = re.search(r"\d{4}-\d{2}", f.stem).group()
        unidad = f.stem.split(periodo)[-1].strip("_") or "Servicio completo"
        archivos.append({"servicio": servicio, "periodo": periodo, "unidad": unidad, "filas": len(t),
                         "columna_vacia": t[sin_nombre].isna().all().all()})
        t = t.drop(columns=sin_nombre)
        t["periodo"] = periodo
        tablas.append(t)
    if not tablas:
        return None, None
    t = pd.concat(tablas, ignore_index=True)
    t.insert(0, "servicio", servicio)
    return t, pd.DataFrame(archivos)

def preparar(t):
    t = t.copy()
    control = {"servicio": t["servicio"].iat[0], "filas": len(t)}

    for original, nuevo in COLS_MONTO.items():
        t[nuevo] = a_monto(t[original])
        control[f"no_conv_{nuevo}"] = (t[nuevo].isna() & t[original].notna()).sum()

    for original, tipo in COLS_HHEE.items():
        texto = t[original].str.strip()
        no_tiene = texto.eq("No tiene")
        partes = texto.str.split(":", n=1, expand=True).reindex(columns=[0, 1]).astype("string")
        monto = a_monto(partes[0])
        horas = pd.to_numeric(
            partes[1].str.replace("hrs", "", regex=False)
                     .str.replace(".", "", regex=False)
                     .str.replace(",", ".", regex=False)
                     .str.strip(),
            errors="coerce")
        # dtype "string" produce números nullable (Float64) y comparaciones con <NA>; se pasan a float común
        t[f"monto_{tipo}"] = monto.where(~no_tiene, 0).astype(float)
        t[f"horas_{tipo}"] = horas.where(~no_tiene, 0).astype(float)
        control[f"vacios_{tipo}"] = texto.isna().sum()
        control[f"otro_formato_{tipo}"] = ((monto.isna() | horas.isna()) & ~no_tiene & texto.notna()).sum()

    for original, nuevo in COLS_FECHA.items():
        t[nuevo] = pd.to_datetime(t[original], format="%d/%m/%Y", errors="coerce")
        control[f"no_conv_{nuevo}"] = (t[nuevo].isna() & t[original].notna()).sum()

    t["cargo"] = t["Cargo o función"].str.strip().str.upper().replace({"CHOFER": "CONDUCTOR"})
    return t, control

In [ ]:
#LECTURA E INTEGRACIÓN DE TODOS LOS SERVICIOS
tablas_serv, archivos_serv, controles = [], [], []
for servicio, patron in SERVICIOS.items():
    t, archivos = leer_servicio(servicio, patron)
    if t is None:
        print(f"{servicio}: sin archivos, se omite")
        continue
    t, control = preparar(t)
    tablas_serv.append(t)
    archivos_serv.append(archivos)
    controles.append(control)

nomina = pd.concat(tablas_serv, ignore_index=True)
archivos_serv = pd.concat(archivos_serv, ignore_index=True)
SERV = list(nomina["servicio"].unique())
columnas = tablas_serv[0].columns
ORIGINALES = list(columnas[1:columnas.get_loc("periodo") + 1])   # 21 columnas del archivo + periodo

print("Servicios cargados:", SERV)
print("Filas de la tabla unida:", len(nomina), "| suma de filas de los archivos:", archivos_serv["filas"].sum())
print("Columna final vacía en todos los archivos:", archivos_serv["columna_vacia"].all())

cobertura = nomina.groupby("servicio").agg(
    archivos=("periodo", "nunique"),
    desde=("periodo", "min"),
    hasta=("periodo", "max"),
    filas=("periodo", "size"),
    personas=("Nombre completo", "nunique"),
).reindex(SERV)
cobertura

In [ ]:
#FILAS POR MES Y SERVICIO
filas_mes = archivos_serv.pivot_table(index="periodo", columns="servicio", values="filas", aggfunc="sum").reindex(columns=SERV)
filas_mes

### 1.1 Ventana común

Los servicios no cubren necesariamente los mismos meses (Centro llega a agosto 2026 y los demás a septiembre 2026). Para que los KPI sean comparables, se calculan sobre los meses presentes en todos los servicios con cobertura completa. Se informa cuántas filas quedan fuera por servicio y mes; no se eliminan de `nomina`, solo se excluyen de `nomina_v`.

**Sur tiene su propia ventana.** El portal no tiene Sur para septiembre y octubre de 2024. Si Sur entrara en la intersección, la ventana de todos bajaría a 22 meses y Centro dejaría de reproducir el `EDA.ipynb` validado en C1. Por eso Sur no recorta la ventana común: se usa en los meses de la ventana que sí tiene (22 de 24). Además:
- **Octubre 2025** está incompleto en Sur (falta el Hospital El Peral y Buin-Paine trae cerca del 40% de sus filas habituales). Se marca como mes atípico.
- **Noviembre 2025** trae cerca de 15% menos filas en los tres hospitales. Se conserva, pero se declara como limitación.

Consecuencia: las comparaciones de Sur con los demás servicios son menos sólidas, en especial la comparación entre años (su primer año tiene 10 meses en vez de 12).

In [ ]:
#VENTANA COMÚN
periodos_por_servicio = nomina.groupby("servicio")["periodo"].apply(set)
VENTANA = sorted(set.intersection(*[p for s, p in periodos_por_servicio.items() if s not in VENTANA_PROPIA]))
print(f"Ventana común: {VENTANA[0]} a {VENTANA[-1]} ({len(VENTANA)} meses)")
for s in VENTANA_PROPIA:
    if s in periodos_por_servicio:
        faltan = [m for m in VENTANA if m not in periodos_por_servicio[s]]
        print(f"{s}: {len(VENTANA) - len(faltan)} de {len(VENTANA)} meses de la ventana; sin datos en {faltan}")

fuera = nomina[~nomina["periodo"].isin(VENTANA)]
print("Filas fuera de la ventana común, por servicio y mes:")
display(fuera.groupby(["servicio", "periodo"]).size().rename("filas").to_frame())

nomina_v = nomina[nomina["periodo"].isin(VENTANA)].copy()
print("Filas antes:", len(nomina), "| después:", len(nomina_v), "| excluidas:", len(nomina) - len(nomina_v))

## 2. Controles de calidad por servicio

Se repiten los controles del `EDA.ipynb` en cada servicio, para detectar problemas de registro propios de cada uno antes de comparar:
1. Valores no convertibles en montos, horas extra y fechas.
2. Valores "-" en remuneraciones adicionales y bonos, por mes.
3. Duplicados exactos.
4. Personas con más de un registro en el mismo mes (unidad de observación).
5. Cambios de estamento entre meses consecutivos (el error de diciembre 2025 en Centro).
6. Horas festivas, derecho a horas extra frente a horas pagadas, región y cargos distintos.
7. Personas con el mismo nombre en más de un servicio.

In [ ]:
#CONVERSIONES POR SERVICIO
pd.DataFrame(controles).set_index("servicio")

In [ ]:
#VALORES "-" POR MES Y SERVICIO (%)
aux = nomina[["servicio", "periodo"]].copy()
aux["guion_rem_adicionales"] = nomina["rem_adicionales"].isna()
aux["guion_bonos"] = nomina["bonos"].isna()
aux["bonos_positivos"] = nomina["bonos"].gt(0)

for col in ["guion_rem_adicionales", "guion_bonos", "bonos_positivos"]:
    print(col)
    display((aux.pivot_table(index="periodo", columns="servicio", values=col, aggfunc="mean")
                .reindex(columns=SERV) * 100).round(1))

In [ ]:
#DUPLICADOS EXACTOS Y REGISTROS POR PERSONA-MES
duplicadas = nomina_v[nomina_v.duplicated(subset=ORIGINALES + ["servicio"], keep=False)]
duplicadas_hhee = (duplicadas["horas_diurnas"] + duplicadas["horas_nocturnas"]) > 0

repeticiones = nomina_v.groupby(["servicio", "periodo", "Nombre completo"]).size()

calidad = pd.DataFrame({
    "copias_adicionales": nomina_v.duplicated(subset=ORIGINALES + ["servicio"]).groupby(nomina_v["servicio"]).sum(),
    "filas_duplicadas_con_hhee": duplicadas_hhee.groupby(duplicadas["servicio"]).sum(),
    "filas": nomina_v.groupby("servicio").size(),
    "persona_mes": repeticiones.groupby("servicio").size(),
    "persona_mes_con_mas_de_un_registro": (repeticiones > 1).groupby("servicio").sum(),
    "max_registros_persona_mes": repeticiones.groupby("servicio").max(),
}).reindex(SERV).fillna(0)
calidad["filas_extra_%"] = (calidad["filas"] - calidad["persona_mes"]) / calidad["filas"] * 100
calidad.round(1)

In [ ]:
#CAMBIOS DE ESTAMENTO ENTRE MESES CONSECUTIVOS, POR SERVICIO
est = (nomina_v.groupby(["servicio", "periodo", "Nombre completo"])["Estamento"].first()
               .reset_index().sort_values(["servicio", "Nombre completo", "periodo"]))
est["estamento_mes_anterior"] = est.groupby(["servicio", "Nombre completo"])["Estamento"].shift(1)
cambios = est.dropna(subset=["estamento_mes_anterior"])
cambios = cambios[cambios["Estamento"] != cambios["estamento_mes_anterior"]]

cambios_mes = (cambios.pivot_table(index="periodo", columns="servicio", values="Estamento", aggfunc="size")
                      .reindex(index=VENTANA, columns=SERV).fillna(0).astype(int))
cambios_mes

In [ ]:
#OTROS CONTROLES POR SERVICIO
tiene_hhee_fila = (nomina_v["horas_diurnas"] + nomina_v["horas_nocturnas"]) > 0
otros = pd.DataFrame({
    "filas_con_horas_festivas": (nomina_v["horas_festivas"] > 0).groupby(nomina_v["servicio"]).sum(),
    "hhee_sin_derecho": (tiene_hhee_fila & nomina_v["Derecho a horas extraordinarias"].str.strip().eq("No"))
                        .groupby(nomina_v["servicio"]).sum(),
    "regiones_distintas": nomina_v.groupby("servicio")["Región"].nunique(),
    "cargos_originales": nomina_v.groupby("servicio")["Cargo o función"].nunique(),
    "cargos_normalizados": nomina_v.groupby("servicio")["cargo"].nunique(),
    "estamentos_distintos": nomina_v.groupby("servicio")["Estamento"].nunique(),
}).reindex(SERV)
display(otros)

print("Derecho a horas extraordinarias frente a horas pagadas (filas):")
pd.crosstab([nomina_v["servicio"], nomina_v["Derecho a horas extraordinarias"]], tiene_hhee_fila)

In [ ]:
#PERSONAS CON EL MISMO NOMBRE EN MÁS DE UN SERVICIO
serv_por_nombre = nomina_v.groupby("Nombre completo")["servicio"].nunique()
print("Nombres presentes en más de un servicio (en algún mes):", (serv_por_nombre > 1).sum())

mismo_mes = nomina_v.groupby(["periodo", "Nombre completo"])["servicio"].nunique()
print("Nombres presentes en más de un servicio en el mismo mes, por mes:")
(mismo_mes > 1).groupby("periodo").sum()

## 3. Tabla persona-mes y KPI por servicio

Se construye la misma tabla persona-mes del `EDA.ipynb`, ahora con clave **(servicio, periodo, nombre)**. La persona se identifica por **(servicio, nombre)**: un mismo nombre en dos servicios se cuenta como dos personas, porque no hay RUT para unirlos (sección 2).

- Estamento: el más frecuente de cada persona dentro de su servicio (igual que `estamento_modal`).
- Marcas: `flag_rem_cero`, `flag_pct_imposible`, `flag_mes_atipico` (según `MESES_ATIPICOS`) y `flag_horas_extremas`, con el límite Q3 + 1,5·RIQ calculado **dentro de cada servicio**.

In [ ]:
#TABLA PERSONA-MES DE TODOS LOS SERVICIOS
CLAVE = ["servicio", "periodo", "Nombre completo"]
pm_all = (
    nomina_v.groupby(CLAVE)
            .agg(
                horas_diurnas=("horas_diurnas", "sum"),
                horas_nocturnas=("horas_nocturnas", "sum"),
                monto_diurnas=("monto_diurnas", "sum"),
                monto_nocturnas=("monto_nocturnas", "sum"),
                rem_bruta=("rem_bruta", "sum"),
            )
            .reset_index()
)
pm_all["horas_hhee"] = pm_all["horas_diurnas"] + pm_all["horas_nocturnas"]
pm_all["monto_hhee"] = pm_all["monto_diurnas"] + pm_all["monto_nocturnas"]
pm_all["tiene_hhee"] = pm_all["horas_hhee"] > 0
pm_all["sobre_40"] = pm_all["horas_diurnas"] > 40
pm_all["pct_hhee_en_rem"] = np.where(pm_all["rem_bruta"] > 0, pm_all["monto_hhee"] / pm_all["rem_bruta"] * 100, np.nan)

idx_persona = pd.MultiIndex.from_frame(pm_all[["servicio", "Nombre completo"]])
estamento_modal = (nomina_v.groupby(["servicio", "Nombre completo"])["Estamento"]
                           .agg(lambda s: s.mode().iloc[0]).replace(NOMBRE_CORTO))
pm_all["estamento"] = estamento_modal.reindex(idx_persona).to_numpy()

# Marcas de calidad
limites = pm_all[pm_all["tiene_hhee"]].groupby("servicio")["horas_hhee"].quantile([0.25, 0.75]).unstack()
limites["limite_extremo"] = limites[0.75] + 1.5 * (limites[0.75] - limites[0.25])
pm_all["flag_mes_atipico"] = [p in MESES_ATIPICOS.get(s, []) for s, p in zip(pm_all["servicio"], pm_all["periodo"])]
pm_all["flag_rem_cero"] = pm_all["rem_bruta"] <= 0
pm_all["flag_pct_imposible"] = pm_all["pct_hhee_en_rem"] > 100
pm_all["flag_horas_extremas"] = pm_all["horas_hhee"] > pm_all["servicio"].map(limites["limite_extremo"])

print("Persona-mes:", len(pm_all))
print("\nLímite de valor extremo por servicio (horas):")
display(limites.reindex(SERV).round(1))
print("Persona-mes marcadas por servicio:")
display(pm_all.groupby("servicio")[[c for c in pm_all.columns if c.startswith("flag_")]].sum().reindex(SERV))

print("Control de la agregación (¿la suma es igual en la nómina y en persona-mes?):")
for col in ["horas_diurnas", "horas_nocturnas", "monto_diurnas", "monto_nocturnas", "rem_bruta"]:
    iguales = nomina_v.groupby("servicio")[col].sum().eq(pm_all.groupby("servicio")[col].sum())
    print(f"{col}: {iguales.all()}")
print("Persona-mes sin estamento asignado:", pm_all["estamento"].isna().sum())

In [ ]:
#PERSISTENCIA POR PERSONA
personas_all = (
    pm_all.groupby(["servicio", "Nombre completo"])
          .agg(
              estamento=("estamento", "first"),
              meses_presente=("periodo", "nunique"),
              meses_con_hhee=("tiene_hhee", "sum"),
          )
          .reset_index()
)
personas_all["pct_meses_con_hhee"] = personas_all["meses_con_hhee"] / personas_all["meses_presente"] * 100
p12_all = personas_all[personas_all["meses_presente"] >= MESES_MINIMOS].copy()
p12_all["persistente"] = (p12_all["pct_meses_con_hhee"] >= UMBRAL_PERSISTENCIA).astype(bool)

# personas con menos de 12 meses no están en p12_all: quedan como no persistentes (False), no como NaN
persistente = p12_all.set_index(["servicio", "Nombre completo"])["persistente"]
pm_all["persistente"] = persistente.reindex(idx_persona).fillna(False).astype(bool).to_numpy()

In [ ]:
#KPI POR SERVICIO
def calcular_kpis(pm, umbral=UMBRAL_PERSISTENCIA):
    por_persona = pm.groupby(["servicio", "Nombre completo"]).agg(
        meses_presente=("periodo", "nunique"),
        meses_con_hhee=("tiene_hhee", "sum"),
    )
    base = por_persona[por_persona["meses_presente"] >= MESES_MINIMOS]
    persistente = base["meses_con_hhee"] / base["meses_presente"] * 100 >= umbral
    peso = pm.loc[pm["tiene_hhee"] & (pm["rem_bruta"] > 0), "pct_hhee_en_rem"]
    con_diurnas = (pm["horas_diurnas"] > 0).sum()
    return pd.Series({
        "persona_mes": len(pm),
        "personas": len(por_persona),
        "personas_12m": len(base),
        "persistentes": persistente.sum(),
        "tasa_persistencia_%": persistente.mean() * 100,
        "frecuencia_%": pm["tiene_hhee"].mean() * 100,
        "peso_mediana_%": peso.median(),
        "mediana_horas_con_hhee": pm.loc[pm["tiene_hhee"], "horas_hhee"].median(),
        "sobre_40_%_con_diurnas": pm["sobre_40"].sum() / con_diurnas * 100 if con_diurnas else np.nan,
        "gasto_hhee_MM": pm["monto_hhee"].sum() / 1e6,
        "gasto_por_persona_mes": pm["monto_hhee"].sum() / len(pm),
    })

def kpis_por_servicio(pm, **kw):
    tabla = pm.groupby("servicio")[pm.columns].apply(lambda g: calcular_kpis(g, **kw)).reindex(SERV)
    tabla.loc["Total"] = calcular_kpis(pm, **kw)
    return tabla

tabla_kpi = kpis_por_servicio(pm_all)
tabla_kpi.round(1)

In [ ]:
#VALIDACIÓN: CENTRO DEBE REPRODUCIR EL EDA.ipynb (solo si la ventana común son los mismos 24 meses)
esperado_centro = {"persona_mes": 43120, "personas": 2430, "personas_12m": 1776, "persistentes": 364}
if "Centro" in SERV:
    print("Ventana común = 2024-09 a 2026-08:", VENTANA[0] == "2024-09" and VENTANA[-1] == "2026-08")
    for k, v in esperado_centro.items():
        obtenido = tabla_kpi.loc["Centro", k]
        print(f"{k}: EDA2 = {obtenido:.0f} | EDA = {v} | coincide: {obtenido == v}")

## 4. Comparación por estamento

Los servicios pueden diferir en su composición por estamento. Como la persistencia cambia mucho entre estamentos (sección 7 del `EDA.ipynb`), una diferencia en la tasa total puede deberse solo a la composición (paradoja de Simpson). Se revisa:
- **Composición** de cada servicio por estamento (proporción de segmento).
- **Persistencia, frecuencia y peso** por estamento y servicio.
- **Tasa de persistencia estandarizada** por estamento: la tasa que tendría cada servicio si tuviera la composición por estamento del total de servicios (estandarización directa).

In [ ]:
#COMPOSICIÓN POR ESTAMENTO (%)
comp_pm = pd.crosstab(pm_all["servicio"], pm_all["estamento"], normalize="index").reindex(index=SERV, columns=ORDEN_EST) * 100
comp_p12 = pd.crosstab(p12_all["servicio"], p12_all["estamento"], normalize="index").reindex(index=SERV, columns=ORDEN_EST) * 100
print("% de persona-mes por estamento:")
display(comp_pm.round(1))
print(f"% de personas con ≥{MESES_MINIMOS} meses por estamento (denominador de persistencia):")
display(comp_p12.round(1))

#FIGURA C1: COMPOSICIÓN POR ESTAMENTO
fig, ax = plt.subplots(figsize=(9, 0.6 * len(SERV) + 1.5))
comp_p12.iloc[::-1].plot(kind="barh", stacked=True, ax=ax, colormap="Blues_r", edgecolor="white", width=0.7)
ax.set_xlabel(f"% de personas con al menos {MESES_MINIMOS} meses")
ax.set_ylabel("")
ax.set_title("Composición por estamento, por servicio")
ax.set_xlim(0, 100)
ax.legend(title="Estamento", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(DATA_DIR / "figura_c1_composicion_estamento.png", dpi=150)
plt.show()

In [ ]:
#KPI POR ESTAMENTO Y SERVICIO
n_est = (p12_all.pivot_table(index="estamento", columns="servicio", values="persistente", aggfunc="size")
                .reindex(index=ORDEN_EST, columns=SERV))
pers_est = (p12_all.pivot_table(index="estamento", columns="servicio", values="persistente", aggfunc="mean")
                   .reindex(index=ORDEN_EST, columns=SERV) * 100)
frec_est = (pm_all.pivot_table(index="estamento", columns="servicio", values="tiene_hhee", aggfunc="mean")
                  .reindex(index=ORDEN_EST, columns=SERV) * 100)
base_peso = pm_all[pm_all["tiene_hhee"] & (pm_all["rem_bruta"] > 0)]
peso_est = (base_peso.pivot_table(index="estamento", columns="servicio", values="pct_hhee_en_rem", aggfunc="median")
                     .reindex(index=ORDEN_EST, columns=SERV))

print(f"Personas con ≥{MESES_MINIMOS} meses, por estamento:")
display(n_est)
print("Tasa de persistencia (%):")
display(pers_est.round(1))
print("Frecuencia de horas extra (% de persona-mes):")
display(frec_est.round(1))
print("Peso de las horas extra en la remuneración bruta (mediana, %):")
display(peso_est.round(1))

In [ ]:
#TASA DE PERSISTENCIA ESTANDARIZADA POR ESTAMENTO
composicion_ref = p12_all["estamento"].value_counts(normalize=True).reindex(ORDEN_EST)

estandarizacion = pd.DataFrame({
    "tasa_bruta_%": p12_all.groupby("servicio")["persistente"].mean().reindex(SERV) * 100,
    "tasa_estandarizada_%": pers_est.mul(composicion_ref, axis=0).sum(min_count=1)
                            / pers_est.notna().mul(composicion_ref, axis=0).sum(),
})
estandarizacion["diferencia_pp"] = estandarizacion["tasa_estandarizada_%"] - estandarizacion["tasa_bruta_%"]
print("Composición de referencia (total de servicios, %):")
print((composicion_ref * 100).round(1))
estandarizacion.round(1)

In [ ]:
#FIGURA C2: PERSISTENCIA POR ESTAMENTO Y SERVICIO
x = np.arange(len(ORDEN_EST))
ancho = 0.8 / len(SERV)

fig, ax = plt.subplots(figsize=(10, 4.5))
for i, s in enumerate(SERV):
    barras = ax.bar(x + (i - (len(SERV) - 1) / 2) * ancho, pers_est[s], width=ancho, color=COLOR[s], label=s)
    ax.bar_label(barras, labels=[f"{v:.0f}" if pd.notna(v) else "" for v in pers_est[s]], fontsize=7, padding=2)
ax.set_xticks(x)
ax.set_xticklabels(ORDEN_EST)
ax.set_ylabel("% de personas persistentes")
ax.set_title(f"Personas con uso persistente de horas extra (≥{UMBRAL_PERSISTENCIA}% de sus meses), por estamento y servicio")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(DATA_DIR / "figura_c2_persistencia_estamento_servicio.png", dpi=150)
plt.show()

## 5. Distribuciones por servicio

Se comparan las tres variables que sostienen los KPI, con las mismas medidas robustas del `EDA.ipynb` (mediana, RIQ, MAD):
- Horas extra por persona-mes, entre quienes tienen horas extra.
- Peso de las horas extra en la remuneración bruta, entre quienes tienen horas extra y remuneración mayor que cero.
- Porcentaje de meses con horas extra por persona (presentes ≥ 12 meses), la variable sobre la que se aplica el umbral.

In [ ]:
#RESUMEN DE DISTRIBUCIONES POR SERVICIO
def resumen_dist(s):
    return pd.Series({
        "n": s.count(),
        "media": s.mean(),
        "mediana": s.median(),
        "desv_est": s.std(),
        "RIQ": s.quantile(0.75) - s.quantile(0.25),
        "MAD": (s - s.median()).abs().median(),
        "p95": s.quantile(0.95),
        "max": s.max(),
    })

con_all = pm_all[pm_all["tiene_hhee"]]
print("Horas extra por persona-mes (con horas extra):")
display(con_all.groupby("servicio")["horas_hhee"].apply(resumen_dist).unstack().reindex(SERV).round(1))
print("Peso de las horas extra en la remuneración bruta, % (con horas extra y rem. > 0):")
display(base_peso.groupby("servicio")["pct_hhee_en_rem"].apply(resumen_dist).unstack().reindex(SERV).round(1))
print(f"% de meses con horas extra por persona (presentes ≥ {MESES_MINIMOS} meses):")
display(p12_all.groupby("servicio")["pct_meses_con_hhee"].apply(resumen_dist).unstack().reindex(SERV).round(1))

In [ ]:
#FIGURA C3: DISTRIBUCIÓN DE HORAS Y PESO POR SERVICIO
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].boxplot([con_all.loc[con_all["servicio"] == s, "horas_hhee"] for s in SERV], showfliers=False)
axes[0].set_title("Horas extra por persona-mes\n(con horas extra)")
axes[0].set_ylabel("Horas en el mes (diurnas + nocturnas)")

axes[1].boxplot([base_peso.loc[base_peso["servicio"] == s, "pct_hhee_en_rem"] for s in SERV], showfliers=False)
axes[1].set_title("Peso de las horas extra en la remuneración bruta\n(con horas extra y rem. > 0)")
axes[1].set_ylabel("% de la remuneración bruta")

for ax in axes:
    ax.set_xticks(range(1, len(SERV) + 1))
    ax.set_xticklabels(SERV)
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(DATA_DIR / "figura_c3_distribucion_horas_peso.png", dpi=150)
plt.show()

In [ ]:
#FIGURA C4: DISTRIBUCIÓN ACUMULADA DEL % DE MESES CON HORAS EXTRA
fig, ax = plt.subplots(figsize=(9, 4.5))
for s in SERV:
    v = np.sort(p12_all.loc[p12_all["servicio"] == s, "pct_meses_con_hhee"])
    ax.step(v, np.arange(1, len(v) + 1) / len(v) * 100, where="post", color=COLOR[s], label=f"{s} (n={len(v)})")
ax.axvline(UMBRAL_PERSISTENCIA, color="black", linestyle="--", linewidth=1)
ax.text(UMBRAL_PERSISTENCIA + 1, 5, f"Umbral {UMBRAL_PERSISTENCIA}%", fontsize=9)
ax.set_xlabel("% de meses con horas extra de cada persona")
ax.set_ylabel("% acumulado de personas")
ax.set_title(f"Distribución acumulada del % de meses con horas extra (personas con ≥{MESES_MINIMOS} meses)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_DIR / "figura_c4_ecdf_persistencia_servicio.png", dpi=150)
plt.show()

## 6. Serie mensual y comparación entre años

- **Serie mensual por servicio** de la frecuencia, los casos sobre 40 horas diurnas, la mediana del monto pagado por hora, el peso y el gasto. El monto por hora sirve como control de registro: un cambio brusco en un mes sugiere un cambio en la forma de informar los montos (como agosto 2025 en Centro).
- **Comparación entre los dos años** (septiembre a agosto), en total y por estamento, para cada servicio.

In [ ]:
#SERIE MENSUAL POR SERVICIO
con_all = pm_all[pm_all["tiene_hhee"]].copy()
con_all["monto_por_hora"] = con_all["monto_hhee"] / con_all["horas_hhee"]

frec_mes = pm_all.pivot_table(index="periodo", columns="servicio", values="tiene_hhee", aggfunc="mean").reindex(columns=SERV) * 100
sobre_40_mes = pm_all.pivot_table(index="periodo", columns="servicio", values="sobre_40", aggfunc="sum").reindex(columns=SERV)
monto_hora_mes = con_all.pivot_table(index="periodo", columns="servicio", values="monto_por_hora", aggfunc="median").reindex(columns=SERV)
peso_mes = base_peso.pivot_table(index="periodo", columns="servicio", values="pct_hhee_en_rem", aggfunc="median").reindex(columns=SERV)
gasto_mes = pm_all.pivot_table(index="periodo", columns="servicio", values="monto_hhee", aggfunc="sum").reindex(columns=SERV) / 1e6

print("Frecuencia mensual (%):");                        display(frec_mes.round(1))
print("Persona-mes con más de 40 horas diurnas:");       display(sobre_40_mes)
print("Mediana del monto pagado por hora ($):");          display(monto_hora_mes.round(0))
print("Mediana del peso en la remuneración bruta (%):");  display(peso_mes.round(1))
print("Gasto mensual en horas extra (millones de $):");   display(gasto_mes.round(1))

In [ ]:
#FIGURA C5: FRECUENCIA MENSUAL POR SERVICIO
fig, ax = plt.subplots(figsize=(10, 4.5))
for s in SERV:
    ax.plot(frec_mes.index, frec_mes[s], marker="o", markersize=3, linewidth=1.5, color=COLOR[s], label=s)
ax.set_title("Frecuencia mensual de horas extraordinarias, por servicio")
ax.set_xlabel("Mes de pago")
ax.set_ylabel("% de persona-mes con horas extra")
ax.tick_params(axis="x", rotation=90)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_DIR / "figura_c5_frecuencia_mensual_servicio.png", dpi=150)
plt.show()

In [ ]:
#FIGURA C6: CONTROLES MENSUALES DE REGISTRO (MAPAS DE CALOR)
# El color se normaliza dentro de cada servicio (columna), para ver meses que se apartan del resto de su propio servicio
paneles = {
    "Casos sobre 40 h diurnas": sobre_40_mes,
    "Mediana monto por hora ($)": monto_hora_mes,
    "Cambios de estamento": cambios_mes,
}
fig, axes = plt.subplots(1, len(paneles), figsize=(4.5 * len(paneles), 0.28 * len(VENTANA) + 2), sharey=True)
for ax, (titulo, tabla) in zip(axes, paneles.items()):
    tabla = tabla.apply(pd.to_numeric, errors="coerce").astype(float)   # las tablas dinámicas pueden quedar con dtype object
    z = (tabla - tabla.median()) / tabla.std().replace(0, np.nan)       # desviación 0 (columna constante) queda sin color
    sbn.heatmap(z, cmap="RdBu_r", center=0, vmin=-3, vmax=3, annot=tabla, fmt=".0f",
                annot_kws={"size": 7}, cbar=False, linewidths=0.5, linecolor="white", ax=ax)
    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.set_ylabel("")
fig.suptitle("Controles mensuales por servicio (color = desviación respecto de la mediana del servicio)")
plt.tight_layout()
plt.savefig(DATA_DIR / "figura_c6_controles_mensuales.png", dpi=150)
plt.show()

In [ ]:
#COMPARACIÓN ENTRE AÑOS (MISMOS MESES), POR SERVICIO
pm_all["anio"] = np.where(pm_all["periodo"] <= "2025-08", "Sep 2024 - ago 2025", "Sep 2025 - ago 2026")

def resumen_anio(t):
    c = t[t["tiene_hhee"]]
    return pd.Series({
        "persona_mes": len(t),
        "frecuencia_%": t["tiene_hhee"].mean() * 100,
        "mediana_horas": c["horas_hhee"].median(),
        "mediana_peso_%": c.loc[c["rem_bruta"] > 0, "pct_hhee_en_rem"].median(),
        "gasto_hhee_MM": t["monto_hhee"].sum() / 1e6,
    })

print("Todos los meses:")
display(pm_all.groupby(["servicio", "anio"])[pm_all.columns].apply(resumen_anio).unstack().reindex(SERV).round(1))
print("Sin meses atípicos (MESES_ATIPICOS):")
sin_atipicos = pm_all[~pm_all["flag_mes_atipico"]]
display(sin_atipicos.groupby(["servicio", "anio"])[sin_atipicos.columns].apply(resumen_anio).unstack().reindex(SERV).round(1))
print("Frecuencia por servicio, estamento y año (%):")
(pm_all.pivot_table(index=["servicio", "estamento"], columns="anio", values="tiene_hhee", aggfunc="mean")
       .reindex(SERV, level=0).mul(100).round(1))

## 7. Gasto y sensibilidad de los KPI

- **Gasto en horas extra** por servicio y estamento, y la parte que corresponde a personas persistentes.
- **Sensibilidad del umbral** de persistencia (50%, 75% y 90%) en cada servicio.
- **KPI con y sin casos marcados**, igual que la sección 6.4 del `EDA.ipynb`, para cada servicio.

In [ ]:
#GASTO EN HORAS EXTRA POR SERVICIO Y ESTAMENTO
gasto = pm_all.groupby("servicio").agg(
    persona_mes=("monto_hhee", "size"),
    gasto_total_MM=("monto_hhee", "sum"),
).reindex(SERV)
gasto["gasto_total_MM"] /= 1e6
gasto["gasto_persistentes_MM"] = pm_all[pm_all["persistente"]].groupby("servicio")["monto_hhee"].sum().reindex(SERV) / 1e6
gasto["%_persistentes"] = gasto["gasto_persistentes_MM"] / gasto["gasto_total_MM"] * 100
gasto["gasto_por_persona_mes"] = gasto["gasto_total_MM"] * 1e6 / gasto["persona_mes"]
gasto["%_rem_bruta"] = pm_all.groupby("servicio")["monto_hhee"].sum() / pm_all.groupby("servicio")["rem_bruta"].sum() * 100
display(gasto.round(1))

print("Gasto por estamento (% del gasto en horas extra de cada servicio):")
gasto_est = (pm_all.pivot_table(index="estamento", columns="servicio", values="monto_hhee", aggfunc="sum")
                   .reindex(index=ORDEN_EST, columns=SERV))
(gasto_est / gasto_est.sum() * 100).round(1)

In [ ]:
#SENSIBILIDAD DEL UMBRAL DE PERSISTENCIA, POR SERVICIO
umbrales = [50, 75, 90]
sens_umbral = pd.DataFrame({
    f"≥{u}%": p12_all.groupby("servicio")["pct_meses_con_hhee"].apply(lambda s: (s >= u).mean() * 100)
    for u in umbrales
}).reindex(SERV)
sens_umbral.loc["Total"] = [(p12_all["pct_meses_con_hhee"] >= u).mean() * 100 for u in umbrales]
display(sens_umbral.round(1))

print("Por estamento y servicio:")
pd.concat({
    f"≥{u}%": p12_all.pivot_table(index="estamento", columns="servicio", values="pct_meses_con_hhee",
                                  aggfunc=lambda s, u=u: (s >= u).mean() * 100).reindex(index=ORDEN_EST, columns=SERV)
    for u in umbrales
}, axis=1).round(1)

In [ ]:
#KPI ANTES Y DESPUÉS DE EXCLUIR CASOS MARCADOS, POR SERVICIO
escenarios = {
    "Original": pm_all,
    "Sin meses atípicos": pm_all[~pm_all["flag_mes_atipico"]],
    "Sin valores imposibles": pm_all[~pm_all["flag_pct_imposible"]],
    "Sin horas extremas": pm_all[~pm_all["flag_horas_extremas"]],
    "Sin ninguna marca": pm_all[~pm_all.filter(like="flag_").any(axis=1)],
}
kpis_escenario = {nombre: kpis_por_servicio(t) for nombre, t in escenarios.items()}

for kpi in ["tasa_persistencia_%", "frecuencia_%", "peso_mediana_%"]:
    print(kpi)
    display(pd.DataFrame({nombre: t[kpi] for nombre, t in kpis_escenario.items()}).T.round(1))

## 8. Resultados de la comparación

**Cobertura.** Se cargaron seis servicios. En la ventana común (septiembre 2024 – agosto 2026) quedan 238.583 filas, 230.053 persona-mes y 9.438 personas presentes al menos 12 meses. Centro reproduce exactamente las cifras del `EDA.ipynb` (43.120 persona-mes, 1.776 personas con ≥12 meses, 364 persistentes). Sur cubre 22 de los 24 meses (sin septiembre ni octubre 2024) y su octubre 2025 está incompleto, por lo que se marca como mes atípico.

**Persistencia.** La tasa total es 16,7% (1.580 de 9.438 personas). Es mayor en Sur (22,7%), Norte (22,1%) y Centro (20,5%), intermedia en Occidente y Sur Oriente (15,7%) y mucho menor en Oriente (8,3%).

**Composición por estamento.** Estandarizar por estamento cambia poco el orden, salvo en Norte: su tasa estandarizada sube a 27,5% (+5,4 pp), porque tiene menos técnicos que el total (13,6% frente a 19,2%) y en Norte los técnicos son muy persistentes (55,4%). Centro baja a 18,8% (−1,7 pp), porque tiene menos médicos (19,7% frente a 25,5%), el estamento con menor persistencia. Oriente queda igual (8,3%): su tasa baja no se explica por composición.

**Patrón por estamento.** Auxiliares y administrativos son los dos estamentos con mayor persistencia en cinco de los seis servicios; en Norte, el segundo lugar lo ocupan los técnicos. La persistencia de los auxiliares va de 19,7% (Oriente) a 76,5% (Norte). Los médicos tienen la menor persistencia en todos los servicios (entre 0,9% y 9,5%). Los técnicos son los que más varían entre servicios: 55,4% en Norte frente a 10,3% en Oriente.

**Intensidad, peso y gasto.** Entre quienes tienen horas extra, la mediana mensual va de 13 horas (Sur Oriente) a 36 horas (Centro), y el peso mediano en la remuneración bruta, de 4,8% (Sur Oriente) a 14,6% (Centro). El gasto total en horas extra en los 24 meses es de $12.397 millones; Centro ($3.427 MM) y Occidente ($3.138 MM) son los mayores. Las personas persistentes concentran entre 39,7% (Oriente) y 80,3% (Norte) del gasto de su servicio.

**Robustez.** Al excluir los casos marcados (meses atípicos, valores imposibles y horas extremas), la tasa de persistencia cambia como máximo 1,5 pp por servicio (Centro: 20,5% → 19,0%; Sur: 22,7% → 21,5%) y el orden entre servicios se mantiene. Con umbrales de 50%, 75% y 90%, Sur, Norte y Centro siguen siendo los tres más altos y Oriente el más bajo.

**Cautelas.**
- *Sur:* la comparación entre años no es directa, porque su primer año tiene 10 meses en vez de 12 (16.323 frente a 19.439 persona-mes). Su gasto del primer año está subestimado.
- *Sur Oriente:* sus horas son mucho más bajas (mediana 13, p95 36), concentra 7.955 de las 10.069 filas con horas festivas y no tiene ninguna fila con horas pagadas sin derecho. Puede ser otra forma de registrar las horas; hay que verificarlo antes de concluir que usa menos horas extra.
- *Occidente:* 13.558 remuneraciones líquidas no convertibles y valores de peso imposibles (máximo 31.815%). Quedan marcados y la mediana es robusta a ellos.
- *Identidad:* sin RUT, 135 nombres aparecen en más de un servicio y se cuentan como personas distintas.

**Qué no permite concluir.** Son asociaciones descriptivas, no causas. Una persistencia alta indica que ciertas personas reciben horas extra de forma recurrente, no prueba por sí sola que falte personal en esos cargos.

## 9. Partición y plan de validación

Corresponde a la parte 6 del enunciado del W2.

### 9.1 Partición por persona

La unidad que se separa es la **persona** (servicio + nombre), no la fila ni la persona-mes. Así una misma persona nunca queda a la vez en entrenamiento y en prueba, y sus meses no se reparten entre conjuntos.

No se separa por meses porque la tasa de persistencia necesita al menos 12 meses por persona: cortar la serie en el tiempo la rompería.

Dentro de cada estrato (servicio × estamento) se desordena a las personas con semilla fija y se asigna 70% a entrenamiento, 15% a validación y 15% a prueba. Así cada conjunto mantiene la misma mezcla de servicios y estamentos.

In [ ]:
#9.1 PARTICIÓN POR PERSONA, ESTRATIFICADA POR SERVICIO Y ESTAMENTO
SEMILLA = 42
PROPORCIONES = {"entrenamiento": 0.70, "validacion": 0.15, "prueba": 0.15}

rng = np.random.default_rng(SEMILLA)
particion = personas_all[["servicio", "Nombre completo", "estamento", "meses_presente"]].copy()
particion["conjunto"] = ""

for _, idx in particion.groupby(["servicio", "estamento"]).groups.items():
    idx = rng.permutation(np.array(idx))
    n_ent = round(len(idx) * PROPORCIONES["entrenamiento"])
    n_val = round(len(idx) * PROPORCIONES["validacion"])
    particion.loc[idx[:n_ent], "conjunto"] = "entrenamiento"
    particion.loc[idx[n_ent:n_ent + n_val], "conjunto"] = "validacion"
    particion.loc[idx[n_ent + n_val:], "conjunto"] = "prueba"

ORDEN_CONJ = list(PROPORCIONES)
resumen_part = pd.DataFrame({
    "personas": particion["conjunto"].value_counts(),
    "personas_12m": particion.loc[particion["meses_presente"] >= MESES_MINIMOS, "conjunto"].value_counts(),
}).reindex(ORDEN_CONJ)
resumen_part["%_personas"] = resumen_part["personas"] / resumen_part["personas"].sum() * 100
resumen_part["%_personas_12m"] = resumen_part["personas_12m"] / resumen_part["personas_12m"].sum() * 100
display(resumen_part.round(1))

### 9.2 Controles de la partición

Se comprueba con código que la partición es correcta:
1. Cada persona aparece una sola vez en la tabla de partición.
2. Ninguna persona quedó en más de un conjunto.
3. Todas las persona-mes recibieron un conjunto.
4. Cuántos nombres que aparecen en más de un servicio quedaron en conjuntos distintos (posible misma persona, ya que no hay RUT).
5. Cuántas persona-mes tiene cada conjunto.
6. Que cada conjunto tenga la misma mezcla de servicios.

In [ ]:
#9.2 CONTROLES DE LA PARTICIÓN: SOLAPAMIENTO Y CONTEOS
CLAVE_P = ["servicio", "Nombre completo"]

print("1. Personas repetidas en la tabla de partición:", particion.duplicated(CLAVE_P).sum())

conj_por_persona = particion.groupby(CLAVE_P)["conjunto"].nunique()
print("2. Personas en más de un conjunto:", (conj_por_persona > 1).sum())

pm_all = pm_all.drop(columns="conjunto", errors="ignore").merge(particion[CLAVE_P + ["conjunto"]], on=CLAVE_P, how="left")
print("3. Persona-mes sin conjunto asignado:", pm_all["conjunto"].isna().sum())

serv_por_nombre = particion.groupby("Nombre completo")["servicio"].nunique()
conj_por_nombre = particion.groupby("Nombre completo")["conjunto"].nunique()
print("4. Nombres en más de un servicio:", (serv_por_nombre > 1).sum(),
      "| de ellos, en conjuntos distintos:", (conj_por_nombre > 1).sum())

pm_conj = pm_all["conjunto"].value_counts().reindex(ORDEN_CONJ)
print("5. Persona-mes por conjunto:")
display(pd.DataFrame({"persona_mes": pm_conj, "%": pm_conj / pm_conj.sum() * 100}).round(1))

print("6. Mezcla de servicios en cada conjunto (% de personas):")
display((pd.crosstab(particion["servicio"], particion["conjunto"], normalize="columns")[ORDEN_CONJ] * 100).round(1))

**Resultado.** No hay personas repetidas ni en más de un conjunto, y todas las persona-mes tienen conjunto. La mezcla de servicios es igual en los tres conjuntos. Limitación: sin RUT, 62 nombres que aparecen en más de un servicio quedaron en conjuntos distintos; si alguno corresponde a la misma persona, habría un traslape de a lo más 0,4% de las personas.

### 9.3 Preprocesamiento ajustado solo con entrenamiento

En la sección 3 el límite de horas extremas (Q3 + 1,5·RIQ por servicio) se calculó con todos los datos. Como es una regla aprendida de los datos, aquí se vuelve a calcular **solo con el conjunto de entrenamiento** y se aplica sin reajustar a validación y prueba. Se compara con el límite calculado con todos los datos para medir cuánto cambiaba.

In [ ]:
#9.3 LÍMITE DE HORAS EXTREMAS AJUSTADO SOLO CON ENTRENAMIENTO
def limite_extremo(t):
    h = t.loc[t["tiene_hhee"], "horas_hhee"]
    q1, q3 = h.quantile(0.25), h.quantile(0.75)
    return q3 + 1.5 * (q3 - q1)

entrenamiento = pm_all[pm_all["conjunto"] == "entrenamiento"]
lim_ent = entrenamiento.groupby("servicio")[["tiene_hhee", "horas_hhee"]].apply(limite_extremo)
lim_todos = pm_all.groupby("servicio")[["tiene_hhee", "horas_hhee"]].apply(limite_extremo)

pm_all["flag_extremo_ent"] = pm_all["horas_hhee"] > pm_all["servicio"].map(lim_ent)

print("Límite de horas extremas por servicio (Q3 + 1,5·RIQ, entre persona-mes con horas extra):")
display(pd.DataFrame({
    "solo_entrenamiento": lim_ent,
    "todos_los_datos": lim_todos,
    "diferencia_horas": lim_todos - lim_ent,
}).reindex(SERV).round(1))

print("% de persona-mes marcadas como extremas con el límite de entrenamiento, por conjunto:")
display((pm_all.pivot_table(index="servicio", columns="conjunto", values="flag_extremo_ent", aggfunc="mean")
               .reindex(index=SERV, columns=ORDEN_CONJ) * 100).round(2))

**Resultado.** El límite aprendido solo con entrenamiento difiere como máximo 3 horas del calculado con todos los datos (idéntico en Centro, Sur y Sur Oriente), por lo que la fuga en el EDA original era pequeña. Aplicado sin reajustar, marca entre 0,1% y 2,2% de las persona-mes en los tres conjuntos; las diferencias entre conjuntos se explican por el menor tamaño de validación y prueba. Desde aquí, el análisis de validación usa `flag_extremo_ent`.

### 9.4 Plan de validación y prueba

**Qué se valida.** El KPI del proyecto es descriptivo (tasa de persistencia), no un modelo predictivo. Por eso la validación no mide precisión de predicción: comprueba que las tasas por estamento y por servicio calculadas en entrenamiento se repiten en personas que no se usaron para calcularlas.

**Regla, fijada antes de mirar validación y prueba.** Para cada estamento y servicio se compara la tasa del conjunto evaluado con la de entrenamiento. La diferencia se acepta si es menor que la variación esperable por azar: 1,96 · √(p·(1−p)/n), con p = tasa de entrenamiento y n = personas del conjunto evaluado (intervalo de confianza de 95% para una proporción).

**Orden de uso.**
1. Entrenamiento: se calculan las tasas de referencia.
2. Validación: se aplica la regla. Si algo falla, se puede revisar el análisis y documentar el cambio.
3. Prueba: se aplica la misma regla **una sola vez**, al final. Su resultado se informa tal cual y no se usa para cambiar umbrales, variables ni definiciones.

**Límite de este holdout.** El umbral de 75%, el mínimo de 12 meses y la definición del KPI se eligieron en W1/C1 mirando todos los datos. Por eso el conjunto de prueba no está completamente "intacto": confirma la estabilidad de las tasas, pero no es una evaluación ciega de esas decisiones.

In [ ]:
#9.4 PLAN DE VALIDACIÓN: REFERENCIA EN ENTRENAMIENTO, REVISIÓN EN VALIDACIÓN, CONFIRMACIÓN ÚNICA EN PRUEBA
p12_part = p12_all.merge(particion[CLAVE_P + ["conjunto"]], on=CLAVE_P, how="left")

def evaluar(conjunto, por):
    ent = p12_part[p12_part["conjunto"] == "entrenamiento"].groupby(por)["persistente"].agg(n="size", tasa="mean")
    ev = p12_part[p12_part["conjunto"] == conjunto].groupby(por)["persistente"].agg(n="size", tasa="mean")
    tolerancia = 1.96 * np.sqrt(ent["tasa"] * (1 - ent["tasa"]) / ev["n"])
    t = pd.DataFrame({
        "tasa_entrenamiento_%": ent["tasa"] * 100,
        f"tasa_{conjunto}_%": ev["tasa"] * 100,
        "diferencia_pp": (ev["tasa"] - ent["tasa"]) * 100,
        "tolerancia_pp": tolerancia * 100,
        f"n_{conjunto}": ev["n"],
    })
    t["dentro_de_tolerancia"] = t["diferencia_pp"].abs() <= t["tolerancia_pp"]
    return t.reindex(ORDEN_EST if por == "estamento" else SERV)

print("VALIDACIÓN, por estamento:")
display(evaluar("validacion", "estamento").round(1))
print("VALIDACIÓN, por servicio:")
display(evaluar("validacion", "servicio").round(1))

In [ ]:
#9.4b PRUEBA: SE EJECUTA UNA SOLA VEZ, AL FINAL, Y NO SE AJUSTA NADA DESPUÉS
print("PRUEBA, por estamento:")
display(evaluar("prueba", "estamento").round(1))
print("PRUEBA, por servicio:")
display(evaluar("prueba", "servicio").round(1))

**Resultado.** En validación y en prueba, las tasas de persistencia por estamento y por servicio quedan dentro de la tolerancia esperable por azar, y el orden se mantiene (auxiliares > administrativos > técnicos > profesionales > médicos; Oriente es el servicio más bajo en los tres conjuntos). El caso más ajustado es Administrativo en validación (+7,0 pp frente a una tolerancia de 7,3 pp, 166 personas). Norte es el servicio con más variación (+5,7 pp en validación, −5,2 pp en prueba) por ser el más pequeño. La prueba se ejecutó una sola vez, después de la validación, y no se modificó ningún umbral ni definición a partir de ella.